# Goals of this notebook
- A cleaned dataframe for analysis
- New columns (revenue, is_return, transaction_type, date_parts)
- Documented decision for every cleaning choice
- A saved cleaned CSV to load in notebook 03,04,05



# Online Retail II - Data Cleaning

## Purpose
Transforming raw data to a cleaned data form ready for analysis :
- Fixing data types
- Adding derived columns ( such as revenue, return flag, transaction types, data parts)
- Normalising inconsistencies
- Documenting decisions on missing data

## Input
`online_retail_II.csv` — raw Kaggle dataset (1,067,371 rows)

## Reference
Finding and rationale documented in `01_data_exploration.ipynb`

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('online_retail_II.csv', parse_dates=['InvoiceDate'])

print(f"Loaded: {df.shape[0]:,} rows * {df.shape[1]} columns")
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Revenue column
df['revenue'] = df['Quantity'] * df['Price']
print(df[['Quantity', 'Price', 'revenue']].head())

   Quantity  Price  revenue
0        12   6.95     83.4
1        12   6.75     81.0
2        12   6.75     81.0
3        48   2.10    100.8
4        24   1.25     30.0


In [ ]:
# Add is_return column
df['is_return'] = (df['Quantity'] < 0) | (df['Invoice'].str.startswith('C'))
print(f"total returns flagged: {df['is_return'].sum():,}")
print(f"percentage of dataset flagged as returns: {df['is_return'].mean() * 100:.2f}%")


total returns flagged: 22,951
percentage of dataset flagged as returns: 2.15%


In [ ]:
# Negative returns to confirm if above code worked correctly
returns_samples = df[df['Quantity'] < 0].head()
print(returns_samples[['Invoice', 'Quantity', 'Price', 'revenue']])

     Invoice  Quantity  Price  revenue
178  C489449       -12   2.95    -35.4
179  C489449        -6   1.65     -9.9
180  C489449        -4   4.25    -17.0
181  C489449        -6   2.10    -12.6
182  C489449       -12   2.95    -35.4


## Derived columns — verification

Three new columns added to support downstream analysis:

**`revenue` = Quantity × Price**
- Positive for sales, negative for returns
- Verified: Row 178 (Invoice C489449) shows -12 units × £2.95 = -£35.40 
- Used for: revenue forecasting, customer monetary value, category-level analysis

**`is_return`** — True/False flag
- True if EITHER Quantity < 0 OR Invoice starts with 'C'
- Captures both standard credit notes AND the 3,457 hidden returns identified in Notebook 01
- Total returns flagged: **22,951 rows (2.15% of dataset)** — matches exploration finding
- Used for: filtering returns in/out depending on analysis question

**`transaction_type`** — Categorical classification
- Distinguishes real product sales from operational entries (shipping, adjustments, discounts, etc.)
- Enables dynamic filtering rather than blanket deletion
- Rationale documented in classification table below

This approach preserves all rows in the master dataset. Each analysis notebook filters based on its specific need.

In [ ]:
# Transaction types
def classify_transactions(stockcode):
    if stockcode in ['POST', 'DOT', 'C2', 'C3']:
        return 'Shipping'
    elif stockcode in ['M', 'm', 'B']:
        return 'Adjustment'
    elif stockcode == 'D':
        return 'Discount'
    elif stockcode == 'S':
        return 'Sample'
    elif stockcode == 'CRUK':
        return 'Charity'
    elif stockcode in ['PADS', 'GIFT']:
        return 'Service'
    else:
        return 'Product'
df['transaction_type'] = df['StockCode'].apply(classify_transactions)
print(df[['StockCode', 'transaction_type']].head())
print(df['transaction_type'].value_counts())

  StockCode transaction_type
0     85048          Product
1    79323P          Product
2    79323W          Product
3     22041          Product
4     21232          Product
transaction_type
Product       1061771
Shipping         3851
Adjustment       1432
Discount          177
Sample            104
Service            20
Charity            16
Name: count, dtype: int64


In [ ]:
# Normalise 'm' to 'M' (safe to re-run)
df['StockCode'] = df['StockCode'].replace('m', 'M')

# Confirm
print(f"'M' count: {(df['StockCode'] == 'M').sum()}")
print(f"'m' remaining: {(df['StockCode'] == 'm').sum()}")



NameError: name 'df' is not defined

In [ ]:
#check if 'm' remaining
print(df['StockCode'].value_counts()[['M']])
print(f"\n any remaining 'm' ? {(df['StockCode'] == 'm').sum()}")

StockCode
M    1426
Name: count, dtype: int64

 any remaining 'm' ? 0


In [ ]:
#Date components for time series analysis
df['year'] = df['InvoiceDate'].dt.year
df['month'] = df['InvoiceDate'].dt.month    
df['quarter'] = df['InvoiceDate'].dt.quarter
df['week'] = df['InvoiceDate'].dt.isocalendar().week
df['day'] = df['InvoiceDate'].dt.day
df['day_of_the_week'] = df['InvoiceDate'].dt.day_name()
df['day_of_week_number'] = df['InvoiceDate'].dt.dayofweek
df['hour'] = df['InvoiceDate'].dt.hour
df['is_weekend'] = df['InvoiceDate'].dt.dayofweek >= 5

#Aggregation periods for time series analysis
df['date'] = df['InvoiceDate'].dt.date
df['Year-Month'] = df['InvoiceDate'].dt.to_period('M')
df['Year-Quarter'] = df['InvoiceDate'].dt.to_period('Q')
df['Year-Week'] = df['InvoiceDate'].dt.to_period('W')

print(df[['InvoiceDate', 'year', 'quarter', 'month', 'week', 'day_of_the_week', 'hour', 'is_weekend', 'Year-Month']].head())



          InvoiceDate  year  quarter  month  week day_of_the_week  hour  \
0 2009-12-01 07:45:00  2009        4     12    49         Tuesday     7   
1 2009-12-01 07:45:00  2009        4     12    49         Tuesday     7   
2 2009-12-01 07:45:00  2009        4     12    49         Tuesday     7   
3 2009-12-01 07:45:00  2009        4     12    49         Tuesday     7   
4 2009-12-01 07:45:00  2009        4     12    49         Tuesday     7   

   is_weekend Year-Month  
0       False    2009-12  
1       False    2009-12  
2       False    2009-12  
3       False    2009-12  
4       False    2009-12  


In [ ]:
df['has_customer_id'] = df['Customer ID'].notna()
print(df['has_customer_id'].value_counts())
print(f"\nMissing: {(~df['has_customer_id']).sum():,} rows ({(~df['has_customer_id']).mean() * 100:.2f}%)")

has_customer_id
True     824364
False    243007
Name: count, dtype: int64

Missing: 243,007 rows (22.77%)


In [ ]:
print("=" * 60)
print("CLEANED DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nData types:")
print(df.dtypes)
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Total revenue: £{df['revenue'].sum():,.2f}")
print(f"Total returns flagged: {df['is_return'].sum():,}")

CLEANED DATASET SUMMARY
Shape: 1,067,371 rows × 29 columns

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'revenue', 'is_return', 'transaction_type', 'year', 'month', 'quarter', 'weekly', 'day', 'day of the week', 'day of week number', 'hour', 'is weekend', 'date', 'Year-Month', 'Year-Quarter', 'Year-Week', 'week', 'day_of_the_week', 'day_of_week_number', 'is_weekend', 'has_customer_id']

Missing values per column:
Invoice                    0
StockCode                  0
Description             4382
Quantity                   0
InvoiceDate                0
Price                      0
Customer ID           243007
Country                    0
revenue                    0
is_return                  0
transaction_type           0
year                       0
month                      0
quarter                    0
weekly                     0
day                        0
day of the week            0
day of week number     

In [ ]:
# Drop duplicate/inconsistent columns from earlier runs
columns_to_drop = [
    'day of the week',
    'day of week number', 
    'is weekend',
    'weekly',
    'Year-Month',
    'Year-Quarter',
    'Year-Week'
]

# Only drop columns that actually exist
existing_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_to_drop)

print(f"Dropped {len(existing_to_drop)} duplicate columns: {existing_to_drop}")
print(f"New shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dropped 7 duplicate columns: ['day of the week', 'day of week number', 'is weekend', 'weekly', 'Year-Month', 'Year-Quarter', 'Year-Week']
New shape: 1,067,371 rows × 22 columns


In [ ]:
# Add clean aggregation period columns for time-series analysis
df['year_month'] = df['InvoiceDate'].dt.to_period('M').astype(str)
df['year_quarter'] = df['InvoiceDate'].dt.to_period('Q').astype(str)
df['year_week'] = df['InvoiceDate'].dt.to_period('W').astype(str)

print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nAll columns:\n{list(df.columns)}")

Final shape: 1,067,371 rows × 25 columns

All columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'revenue', 'is_return', 'transaction_type', 'year', 'month', 'quarter', 'day', 'hour', 'date', 'week', 'day_of_the_week', 'day_of_week_number', 'is_weekend', 'has_customer_id', 'year_month', 'year_quarter', 'year_week']


In [ ]:
output_path = 'online_retail_II_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"✓ Saved: {output_path}")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"  File size: {file_size_mb:.1f} MB")

✓ Saved: online_retail_II_cleaned.csv
  Rows: 1,067,371
  Columns: 25
  File size: 200.6 MB
